# Causal language model

Causal language modeling (AKA autoregressive modeling) predicts the token in a sequence of tokens, and the model can only attend to tokens on the left. This means the model cannot see future tokens. Used by decode models like GPT. this approach predicts the next token based on all previous tokens in the left to predict the next token.

In [ ]:
!pip install transformers datasets evaluate

## Load ELI5 dataset

Start by loading the first 5000 examples from ELI5-Category dataset with the hugging face datasets libary.

In [ ]:
from datasets import load_dataset

eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")

Split the dataset's <b>train</b> split into a train and test set with the <b>train_text_split</b> method:

In [3]:
eli5 = eli5.train_test_split(test_size=0.2)
eli5 = eli5.flatten()
eli5["train"][0]

{'q_id': '7dafkf',
 'title': 'How do we know that trees and plants do not experience pain when we cut them?',
 'selftext': 'Do they?',
 'category': 'Biology',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dpwatkv', 'dpwmf3j'],
 'answers.text': ["No, plants don't feel pain. Science hasn't found any evidence that plants feel pain, as they lack brains and a nervous system. It's agreed upon that you need those things for the more complicated sensations, such as pain. Some plants do respond to stuff, like Venus flytraps catching flies, but it's not the same thing. Plants are simple things, pain is too complicated of a response.",
  'Mammals use pain to avoid danger, by making the creature not want to experience that feeling. But plants however, are incapable of moving, and do not have brains, so they have no use for pain.'],
 'answers.score': [4, 3],
 'answers.text_urls': [[], []],
 'title_urls': ['url'],
 'selftext_urls': ['url']}

While this looks like a lot, we are only interested in the <b>text</b> field. What's cool about language modeling tasks is you don't need labels(also knows as an unsupervised task) because the next word is the label.

## Preprocess
The next step is to load a DIstilGPT2 tokenizer to process the <b>text</b> subfield.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

You’ll notice from the example above, the text field is actually nested inside answers. This means you’ll need to extract the text subfield from its nested structure with the flatten method.

Each subfield is now a separate column as indicated by the answers prefix, and the text field is a list now. Instead of tokenizing each sentence separately, convert the list to a string so we can jointly tokenize them.

Here is a first preprocessing function to join the list of strings for each example and tokenize the result:

In [5]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

To apply this preprocessing function over the entire dataset, lets use the 🤗 Datasets map method. You can speed up the map function by setting batched=True to process multiple elements of the dataset at once, and increasing the number of processes with num_proc. Remove any columns you don’t need:

In [6]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4051 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1985 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1478 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1197 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1071 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1374 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1145 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1947 > 1024). Running this sequence through the model will result in indexing errors


In [7]:
block_size = 128

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

Apply the <b>group_text</b> function over the entire dataset:

In [8]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

Now create a batch of examplesusing <b>DataCollatorForLanguageModeling</b>. Its more efficient to dynamically pad the sentence to the longest length in batch during collation, instead of padding the whole dataset to the maximum length.

Use the end-of-sequence token as the padding token and set <b>mlm=False</b>. This will use the inputs as labels shifted to the right by one element:

In [9]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Train

Now we are ready to start training our model! Load DistilGPT2 with AutoModelForCausalLM:

In [10]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

At this point, only three steps remain:

- Define your training hyperparameters in <b>TrainingArguments</b>. The only required parameter is output_dir which specifies where to save your model. You’ll push this model to the Hub by setting push_to_hub=True (you need to be signed in to Hugging Face to upload your model).
- Pass the training arguments to <b>Trainer</b> along with the model, datasets, and data collator.
- Call <b>train()</b> to finetune your model.

In [11]:
training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.912302,3.809236
2,3.825282,3.799181
3,3.784404,3.796978


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3900, training_loss=3.845782752403846, metrics={'train_runtime': 854.4323, 'train_samples_per_second': 36.512, 'train_steps_per_second': 4.564, 'total_flos': 1018959341027328.0, 'train_loss': 3.845782752403846, 'epoch': 3.0})

In [12]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
3.784404,3.796978,3


Perplexity: 44.57


In [ ]:
from pathlib import Path

model_path = Path("<relative_path>/my_awesome_eli5_clm-model")
model_path.mkdir(parents=True, exist_ok=True)
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to {model_path.resolve()}")


## Inference

In [13]:
prompt = "Somatic hypermutation allows the immune system to"

In [19]:
from transformers import pipeline

generator = pipeline("text-generation", model=str(model_path), tokenizer=tokenizer)
result = generator(prompt, max_new_tokens=50, do_sample=True, temperature=0.8)
result[0]["generated_text"]


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

"Somatic hypermutation allows the immune system to survive and replicate. The immune system in the body recognizes the danger of starvation and even is left behind by the death of the body. Humans and animals also lack the ability to effectively reproduce. So if you die in a war, it's because of"

In [25]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(str(model_path))
inputs = tokenizer(prompt, return_tensors="pt").input_ids

In [26]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(str(model_path))
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [27]:
tokenizer.batch_decode(outputs, skip_special_tokens=True)

['Somatic hypermutation allows the immune system to adapt to a more drastic response. When the immune system is overwhelmed, the immune system does not adapt to new stimuli (such as allergies, etc). The reason is that the immune system reacts to these changes is that they create new signals to the body that can\'t be changed for fear of it (the "invasive" response is the "tactin" response that the immune system has not gotten through with) so when one of these messages comes out, the immune system adapts to changes']